# Demanda eléctrica por área del SEIN: la prueba de 2026

**TB1 · Metodología para Data Science (1ACC0240) · UPC · ciclo 2026-2 · Grupo 3**

Notebook 03 de CRISP-DM para el caso peruano: **evaluación en la prueba fuera de distribución**, de enero a octubre de
2026, con El Niño Costero extraordinario (el ICEN llegó a 3.98 en agosto). Es el resultado central del paper.

**Reglas de esta corrida** (spec §8.1):

- **Aquí no se elige nada.** El modelo (Random Forest por área), sus hiperparámetros, las variantes de exógenas y las
  políticas de reentrenamiento salen del [notebook 02](02_sein_modelado_evaluacion.ipynb), elegidos en 2025. Como
  alternativa se reporta el XGBoost global, que empató con él en 2025.
- **Las hipótesis se escribieron antes de mirar 2026** (notebook 02, §12) y se evalúan tal como están, en la §9.
- **Se corre una sola vez.** Si algo falla a mitad de camino se corrige el código, no la elección.

| Paso | Qué hace |
|---|---|
| 1. Datos | Los datasets de 2016 a octubre de 2026, con la prueba incluida |
| 2. Referencias ingenuas en 2026 | La vara de la prueba: la mejor de las tres en cada área y horizonte |
| 3. El modelo fijo frente a las referencias | Entrenado hasta 2025, sin tocarlo durante 2026 (política A) |
| 4. Ablación de exógenas en 2026 | M0 a M3 con el modelo fijo |
| 5. Experimento de adaptación | Fijo frente a reentrenado cada semana, con y sin exógenas, y con la alerta de ENFEN |
| 6. Mes a mes | Cómo cambia el error a medida que El Niño se intensifica |
| 7. Regímenes | Categoría de El Niño, horas de mayor demanda y días con temperatura fuera de lo visto |
| 8. Diagnóstico | Real contra predicho en el pico de El Niño |
| 9. Las hipótesis, una por una | Se cumplen o no, con su evidencia |
| 10. Guardar | Predicciones y tablas para la app y el paper |

Corre en Colab en unas 3 horas (el Random Forest se reentrena cada semana en la CPU). Con `TPF_RAPIDO=1` corre en
minutos, pero **solo con datos sintéticos**: con los datos reales mostraría la prueba antes de tiempo.

# Configuración

In [ ]:
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and "TETOUAN_DATA" not in os.environ:
    RAMA = "develop"
    !pip install -q "git+https://github.com/R0SEWT/tetouan-power-forecast@{RAMA}"
    from tetouan_power_forecast.medallon import usar_drive
    usar_drive()
elif not EN_COLAB:
    os.environ.setdefault("TETOUAN_DATA", str(Path.cwd().parent / "data"))

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import r2_score

from tetouan_power_forecast.coes import cargar_demanda_areas
from tetouan_power_forecast.evaluacion import bootstrap_diferencia_mae, errores, hora_objetivo, mae_por_grupo
from tetouan_power_forecast.exogenas import icen_disponible
from tetouan_power_forecast.medallon import raiz_datos
from tetouan_power_forecast.modelado_sein import (
    VARIANTES_ABLACION,
    bloques,
    categoria_icen,
    en,
    es_hora_pico,
    gradient_boosting_global,
    horas_mayor_demanda,
    mejor_referencia,
    predecir_global,
    predecir_local,
    predecir_por_bloques,
    preparar_datasets,
    random_forest,
    subestimacion_sobre_p90,
    tramo,
)
from tetouan_power_forecast.modelos import ModeloRazon
from tetouan_power_forecast.variables_sein import AREAS, HORIZONTES, PASO, REFERENCIAS

RAIZ = raiz_datos()
RAPIDO = os.environ.get("TPF_RAPIDO") == "1"  # solo con datos sintéticos
SEMILLA = 0
FRECUENCIA_REENTRENO = "MS" if RAPIDO else "W-MON"  # cada lunes a las 00:00, como en 2025
INICIO_PRUEBA = pd.Timestamp("2026-01-01")
NOMBRES = {a: a.capitalize() for a in AREAS}
NOMBRES_REF = {"persistencia": "Persistencia", "ayer": "Ayer", "semana_pasada": "Semana pasada"}
COLORES = {"NORTE": "#E69F00", "CENTRO": "#0072B2", "SUR": "#009E73"}
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", message=".*mismatched devices.*")

# Lo elegido en 2025 en el notebook 02. No se cambia aquí.
ELEGIDO = "Random Forest por área"
ALTERNATIVO = "Gradient Boosting global"
HIPERPARAMETROS = {
    ELEGIDO: {"n_estimators": 100, "min_samples_leaf": 5, "max_samples": 0.3, "max_features": 0.3, "max_depth": 20},
    ALTERNATIVO: {"colsample_bytree": 0.7, "learning_rate": 0.0358, "max_depth": 10, "min_child_weight": 20,
                  "n_estimators": 800, "reg_lambda": 1.0, "subsample": 0.85},
}
MODELOS = {
    ELEGIDO: ModeloRazon(random_forest(SEMILLA, **HIPERPARAMETROS[ELEGIDO])),
    ALTERNATIVO: gradient_boosting_global(SEMILLA, **HIPERPARAMETROS[ALTERNATIVO]),
}
metadatos_02 = RAIZ / "modelos" / "sein" / "modelo_final.json"
if metadatos_02.exists():  # el modelo y los hiperparámetros tienen que ser los del notebook 02
    guardado = json.loads(metadatos_02.read_text())
    assert guardado["modelo"] == ELEGIDO, guardado["modelo"]
    assert guardado["hiperparametros"] == HIPERPARAMETROS[ELEGIDO], guardado["hiperparametros"]
    print("Modelo e hiperparámetros verificados contra el notebook 02")
else:
    print("Sin modelo_final.json: se usan los hiperparámetros escritos arriba")
print("Elegido:", ELEGIDO, "· alternativa:", ALTERNATIVO, "· RAPIDO =", RAPIDO)

# 1. Datos

Los mismos datasets del notebook 02 (`preparar_datasets`, con la regla de información), ahora con la prueba
incluida. Entrenamiento 2016-2024, validación 2025 y prueba 2026, con un hueco de H pasos antes de cada corte. El
modelo fijo se entrena con entrenamiento y validación ("desarrollo") y no se toca durante 2026.

In [ ]:
demanda = cargar_demanda_areas()
silver_exo = RAIZ / "exogenas" / "silver"
exogenas = {
    "temperatura": pd.read_parquet(silver_exo / "temperatura_areas.parquet"),
    "icen": pd.read_parquet(silver_exo / "icen.parquet"),
    "nino12": pd.read_parquet(silver_exo / "nino12_semanal.parquet"),
    "eventos": pd.read_parquet(silver_exo / "eventos_coes.parquet"),
}
try:
    import holidays
    FERIADOS = list(holidays.PE(years=range(2016, 2027)))
except ImportError:
    FERIADOS = []
    print("sin el paquete holidays: los feriados quedan en cero")

datos, filas = {}, []
for nombre_h, h in HORIZONTES.items():
    locales, global_ = preparar_datasets(demanda, h, variantes=VARIANTES_ABLACION, feriados=FERIADOS, **exogenas)
    datos[nombre_h] = {"locales": locales, "global": global_}
    for a in AREAS:
        conteo = locales[a]["particion"].value_counts()
        prueba = tramo(locales[a], "prueba")
        filas.append({"Horizonte": nombre_h, "Área": NOMBRES[a], "Desarrollo": len(tramo(locales[a], "desarrollo")),
                      "Prueba": conteo.get("prueba", 0), "Prueba desde": prueba["DateTime"].min(),
                      "Prueba hasta": prueba["DateTime"].max()})
display(pd.DataFrame(filas).set_index(["Horizonte", "Área"]))


def ajustar_y_predecir(nombre_h, columnas, entrenar, evaluar, metodo=ELEGIDO):
    # Predicciones por área de un método entrenado y evaluado en las filas indicadas
    if metodo.endswith("global"):
        return predecir_global(MODELOS[metodo], datos[nombre_h]["global"], columnas, entrenar, evaluar)
    return predecir_local(MODELOS[metodo], datos[nombre_h]["locales"], columnas, entrenar, evaluar)


def prueba(nombre_h, a):
    return tramo(datos[nombre_h]["locales"][a], "prueba")

# 2. Referencias ingenuas en 2026

Persistencia, ayer y semana pasada a la misma hora. El modelo se compara con **la mejor de las tres en la prueba**,
en cada área y horizonte (spec §8.2).

In [ ]:
filas = []
for nombre_h in HORIZONTES:
    for a in AREAS:
        t = prueba(nombre_h, a)
        for ref in REFERENCIAS:
            e = errores(t["real"], t[ref])
            filas.append({"Horizonte": nombre_h, "Área": NOMBRES[a], "Referencia": NOMBRES_REF[ref],
                          "MAE (MW)": e["MAE (kW)"], "MAPE (%)": e["MAPE (%)"]})
display(pd.DataFrame(filas).pivot_table(index=["Horizonte", "Referencia"], columns="Área",
                                        values=["MAE (MW)", "MAPE (%)"]).round(2))
MEJOR_REF = {h: {a: mejor_referencia(prueba(h, a)) for a in AREAS} for h in HORIZONTES}
for h in HORIZONTES:
    print(f"Mejor referencia en 2026 a {h}:", {NOMBRES[a]: NOMBRES_REF[r] for a, r in MEJOR_REF[h].items()})


def ref_mejor(nombre_h, a):
    return prueba(nombre_h, a)[MEJOR_REF[nombre_h][a]].to_numpy()

# 3. El modelo fijo frente a las referencias

El modelo elegido, entrenado hasta 2025 y sin reentrenar en 2026 (política A, M0). Se reporta también el XGBoost
global. La diferencia de MAE con la mejor referencia va con su intervalo al 95 % por *bootstrap* de días.

In [ ]:
inicio = time.time()
fijo = {h: {v: ajustar_y_predecir(h, v, en("desarrollo"), en("prueba")) for v in VARIANTES_ABLACION} for h in HORIZONTES}
alternativo = {h: ajustar_y_predecir(h, "M0", en("desarrollo"), en("prueba"), metodo=ALTERNATIVO) for h in HORIZONTES}
print(f"{(time.time() - inicio) / 60:.1f} min")

filas = []
for h in HORIZONTES:
    for a in AREAS:
        t = prueba(h, a)
        mae_ref = errores(t["real"], ref_mejor(h, a))["MAE (kW)"]
        for metodo, pred in [(f"Mejor referencia ({NOMBRES_REF[MEJOR_REF[h][a]]})", ref_mejor(h, a)),
                             (f"{ELEGIDO}, fijo (A)", fijo[h]["M0"][a]), (f"{ALTERNATIVO}, fijo", alternativo[h][a])]:
            e = errores(t["real"], pred)
            fila = {"Horizonte": h, "Área": NOMBRES[a], "Método": metodo, "MAE (MW)": e["MAE (kW)"],
                    "RMSE (MW)": e["RMSE (kW)"], "MAPE (%)": e["MAPE (%)"], "R²": r2_score(t["real"], pred),
                    "Mejora vs. mejor referencia (%)": 100 * (1 - e["MAE (kW)"] / mae_ref)}
            if not metodo.startswith("Mejor"):
                dif, bajo, alto = bootstrap_diferencia_mae(t["real"], pred, ref_mejor(h, a), t["DateTime"], semilla=SEMILLA)
                fila.update({"Δ MAE vs. referencia (MW)": dif, "IC 95 % bajo": bajo, "IC 95 % alto": alto})
            filas.append(fila)
frente_referencias = pd.DataFrame(filas)
display(frente_referencias.set_index(["Horizonte", "Área", "Método"]).round(2))

# 4. Ablación de exógenas en 2026

Con el modelo fijo: M0 (demanda y calendario), M1 (+ clima), M2 (+ El Niño) y M3 (+ fallas en curso). Las fallas se
mantienen definidas como en 2025 para poder comparar con la validación, y por eso se reporta también M2, que no las
lleva.

In [ ]:
CONTRASTES_ABLACION = [("M1", "M0", "Clima (M1 − M0)"), ("M2", "M1", "El Niño (M2 − M1)"),
                       ("M3", "M2", "Fallas (M3 − M2)"), ("M3", "M0", "Todas (M3 − M0)"), ("M2", "M0", "Sin fallas (M2 − M0)")]
filas = []
for h in HORIZONTES:
    for a in AREAS:
        t = prueba(h, a)
        for nuevo, base, nombre in CONTRASTES_ABLACION:
            dif, bajo, alto = bootstrap_diferencia_mae(t["real"], fijo[h][nuevo][a], fijo[h][base][a], t["DateTime"],
                                                       semilla=SEMILLA)
            filas.append({"Horizonte": h, "Contraste": nombre, "Área": NOMBRES[a], "Δ MAE (MW)": dif,
                          "IC 95 % bajo": bajo, "IC 95 % alto": alto, "Distinguible": not (bajo <= 0 <= alto)})
ablacion = pd.DataFrame(filas)
display(pd.DataFrame({v: {(h, NOMBRES[a]): errores(prueba(h, a)["real"], fijo[h][v][a])["MAE (kW)"]
                          for h in HORIZONTES for a in AREAS} for v in VARIANTES_ABLACION}).round(2)
        .style.set_caption("MAE (MW) en 2026 del modelo fijo, por variante"))
display(ablacion.set_index(["Horizonte", "Contraste", "Área"]).round(2))

# 5. Experimento de adaptación

Las mismas políticas que en 2025 (spec §8.4), ahora durante El Niño extraordinario:

| | Sin exógenas (M0) | Con exógenas (M3) |
|---|---|---|
| **Modelo fijo** | A | B |
| **Reentrenado cada semana** | C | D |

Más reentrenar cuando cambia la categoría del ICEN que ya se conocía (la alerta de ENFEN), sola y sumada a la
semanal, y la variante M2 (sin fallas) fija y semanal. Ventana creciente, hiperparámetros congelados y un hueco de H
pasos antes de cada reentrenamiento.

In [ ]:
POLITICAS = {"fijo": "fijo", "semanal": "periodica", "alerta ENFEN": "alerta", "semanal + alerta": "periodica+alerta"}
VARIANTES_POLITICA = {"fijo": ["M0", "M2", "M3"], "semanal": ["M0", "M2", "M3"], "alerta ENFEN": ["M0", "M3"],
                      "semanal + alerta": ["M0", "M3"]}
adaptacion, pred_adaptacion = [], {}
for h, pasos in HORIZONTES.items():
    inicio = time.time()
    fechas_prueba = tramo(datos[h]["global"], "prueba")["DateTime"].drop_duplicates().sort_values()
    alerta = categoria_icen(icen_disponible(fechas_prueba, exogenas["icen"]))
    pred_adaptacion[h] = {}
    for nombre_p, politica in POLITICAS.items():
        cortes = bloques(fechas_prueba, politica, frecuencia=FRECUENCIA_REENTRENO, alerta=alerta)
        for v in VARIANTES_POLITICA[nombre_p]:
            if nombre_p == "fijo":
                pred = fijo[h][v]  # ya calculado en la §3: un solo bloque, entrenado hasta el hueco
            else:
                pred = predecir_por_bloques(lambda e, s, hh=h, vv=v: ajustar_y_predecir(hh, vv, e, s), cortes,
                                            pasos * PASO, seleccionar=en("prueba"))
            pred_adaptacion[h][(nombre_p, v)] = pred
            for a in AREAS:
                t = prueba(h, a)
                e = errores(t["real"], pred[a])
                adaptacion.append({"Horizonte": h, "Política": nombre_p, "Variante": v, "Área": NOMBRES[a],
                                   "Reentrenamientos": len(cortes) - 1, "MAE (MW)": e["MAE (kW)"],
                                   "MAPE (%)": e["MAPE (%)"],
                                   "Mejora vs. mejor referencia (%)": 100 * (1 - e["MAE (kW)"] / errores(t["real"], ref_mejor(h, a))["MAE (kW)"])})
        print(f"{h} · {nombre_p}: {len(cortes) - 1} reentrenamientos · {(time.time() - inicio) / 60:.1f} min", flush=True)
adaptacion = pd.DataFrame(adaptacion)
display(adaptacion.pivot_table(index=["Horizonte", "Variante", "Política"], columns="Área",
                               values=["MAE (MW)", "Mejora vs. mejor referencia (%)"]).round(2))

In [ ]:
CONTRASTES_ADAPTACION = {
    "Reentrenar sin exógenas (C − A)": (("semanal", "M0"), ("fijo", "M0")),
    "Reentrenar con exógenas (D − B)": (("semanal", "M3"), ("fijo", "M3")),
    "Reentrenar sin fallas (semanal M2 − fijo M2)": (("semanal", "M2"), ("fijo", "M2")),
    "Exógenas con reentrenamiento (D − C)": (("semanal", "M3"), ("semanal", "M0")),
    "Alerta en vez de semanal (alerta − semanal, M3)": (("alerta ENFEN", "M3"), ("semanal", "M3")),
    "Alerta además de semanal (semanal + alerta − semanal, M3)": (("semanal + alerta", "M3"), ("semanal", "M3")),
}
filas = []
for h in HORIZONTES:
    p = pred_adaptacion[h]
    for nombre, (nuevo, base) in CONTRASTES_ADAPTACION.items():
        for a in AREAS:
            t = prueba(h, a)
            dif, bajo, alto = bootstrap_diferencia_mae(t["real"], p[nuevo][a], p[base][a], t["DateTime"], semilla=SEMILLA)
            filas.append({"Horizonte": h, "Contraste": nombre, "Área": NOMBRES[a], "Δ MAE (MW)": dif,
                          "IC 95 % bajo": bajo, "IC 95 % alto": alto, "Distinguible": not (bajo <= 0 <= alto)})
efectos_adaptacion = pd.DataFrame(filas)
display(efectos_adaptacion.set_index(["Horizonte", "Contraste", "Área"]).round(2))

# 6. Mes a mes

El error de cada mes de 2026 con el modelo fijo (A), el reentrenado cada semana (C y D) y la mejor referencia. Si
El Niño saca la demanda de lo visto, el fijo debería empeorar a medida que avanza el año y el reentrenado aguantar
mejor.

In [ ]:
filas = []
for h in HORIZONTES:
    for a in AREAS:
        t = prueba(h, a)
        mes = (t["DateTime"] + HORIZONTES[h] * PASO).dt.to_period("M").astype(str).to_numpy()
        tabla = mae_por_grupo(t["real"], {"Fijo (A)": pred_adaptacion[h][("fijo", "M0")][a],
                                          "Semanal (C)": pred_adaptacion[h][("semanal", "M0")][a],
                                          "Semanal con exógenas (D)": pred_adaptacion[h][("semanal", "M3")][a],
                                          "Mejor referencia": ref_mejor(h, a)}, mes)
        filas.append(tabla.assign(Horizonte=h, Área=NOMBRES[a]).rename_axis("Mes").reset_index())
mensual = pd.concat(filas, ignore_index=True)
mensual["Mejora A (%)"] = 100 * (1 - mensual["Fijo (A)"] / mensual["Mejor referencia"])
mensual["Mejora C (%)"] = 100 * (1 - mensual["Semanal (C)"] / mensual["Mejor referencia"])
display(mensual.pivot_table(index="Mes", columns=["Horizonte", "Área"], values=["Mejora A (%)", "Mejora C (%)"]).round(1))

fig, ejes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for i, h in enumerate(HORIZONTES):
    for eje, a in zip(ejes[i], AREAS):
        d = mensual[(mensual["Horizonte"] == h) & (mensual["Área"] == NOMBRES[a])]
        for col, estilo in [("Fijo (A)", "-o"), ("Semanal (C)", "--s"), ("Semanal con exógenas (D)", ":^"),
                            ("Mejor referencia", "-")]:
            eje.plot(d["Mes"].str[5:], d[col], estilo, ms=3, label=col,
                     color="grey" if col == "Mejor referencia" else COLORES[a], alpha=1 if col != "Semanal con exógenas (D)" else 0.6)
        eje.set_title(f"{NOMBRES[a]} a {h}")
    ejes[i][0].set_ylabel("MAE (MW)")
ejes[0][0].legend(fontsize=8)
plt.suptitle("Error mes a mes en 2026")
plt.tight_layout()
plt.show()

# 7. Regímenes

Con el modelo fijo (A) y el reentrenado cada semana (C):

- **El Niño:** la categoría del ICEN definitivo del mes que se predice (septiembre y octubre todavía no tienen ICEN
  publicado y quedan aparte).
- **Horas de mayor demanda:** las cuatro horas con más demanda media de cada área en 2026, y la fracción de esas horas
  en que el modelo subestima más que el p90 del error de la mejor referencia.
- **Temperatura fuera de lo visto** (hipótesis 4): los días en que la temperatura media del área supera la máxima
  de ese mes entre 2016 y 2025. Los árboles no extrapolan, así que ahí debería fallar más.

In [ ]:
ICEN_DEFINITIVO = exogenas["icen"].set_index(exogenas["icen"]["mes"].dt.to_period("M"))["icen"]


def dias_fuera_de_rango(area):
    # Días de 2026 con temperatura media mayor que la máxima del mismo mes entre 2016 y 2025
    diaria = exogenas["temperatura"].set_index("fecha_hora")[area.lower()].resample("D").mean().dropna()
    historia = diaria[diaria.index < INICIO_PRUEBA]
    tope = historia.groupby(historia.index.month).max()
    nueva = diaria[diaria.index >= INICIO_PRUEBA]
    return pd.DatetimeIndex(nueva.index[nueva.to_numpy() > nueva.index.month.map(tope).to_numpy()])


filas = []
for h in HORIZONTES:
    adelanto = HORIZONTES[h] * PASO
    for a in AREAS:
        t = prueba(h, a).reset_index(drop=True)
        objetivo = t["DateTime"] + adelanto
        preds = {"Fijo (A)": pred_adaptacion[h][("fijo", "M0")][a], "Semanal (C)": pred_adaptacion[h][("semanal", "M0")][a],
                 "Mejor referencia": ref_mejor(h, a)}
        pico = es_hora_pico(t["DateTime"], horas_mayor_demanda(t, adelanto), adelanto)
        fuera = objetivo.dt.normalize().isin(dias_fuera_de_rango(a)).to_numpy()
        icen_mes = objetivo.dt.to_period("M").map(ICEN_DEFINITIVO)
        regimenes = {
            "El Niño (ICEN del mes)": np.where(icen_mes.isna(), "sin ICEN publicado", categoria_icen(icen_mes).astype(str)),
            "Horas de mayor demanda": np.where(pico, "Mayor demanda", "Resto"),
            "Temperatura": np.where(fuera, "Fuera de lo visto", "Dentro de lo visto"),
        }
        for corte, etiqueta in regimenes.items():
            tabla = mae_por_grupo(t["real"], preds, etiqueta)
            for grupo, fila in tabla.iterrows():
                filas.append({"Horizonte": h, "Corte": corte, "Grupo": grupo, "Área": NOMBRES[a],
                              "Periodos": int((etiqueta == grupo).sum()), **fila.to_dict(),
                              "Mejora A (%)": 100 * (1 - fila["Fijo (A)"] / fila["Mejor referencia"]),
                              "Mejora C (%)": 100 * (1 - fila["Semanal (C)"] / fila["Mejor referencia"])})
        for nombre in ("Fijo (A)", "Semanal (C)"):
            filas.append({"Horizonte": h, "Corte": "Horas de mayor demanda",
                          "Grupo": f"Subestimación > p90 de la referencia, {nombre} (%)", "Área": NOMBRES[a],
                          "Periodos": int(pico.sum()),
                          "Fijo (A)": 100 * subestimacion_sobre_p90(t["real"], preds[nombre], preds["Mejor referencia"], pico)})
regimenes_2026 = pd.DataFrame(filas)
display(regimenes_2026.set_index(["Horizonte", "Corte", "Grupo", "Área"]).round(2))
print("Días con temperatura fuera de lo visto en 2026:", {NOMBRES[a]: len(dias_fuera_de_rango(a)) for a in AREAS})

# 8. Diagnóstico

Dos semanas de julio de 2026, en el pico de El Niño, en el Norte: lo real, el modelo fijo, el reentrenado cada semana
y la mejor referencia, a 1 h y a 24 h. Y los residuos del modelo fijo en toda la prueba.

In [ ]:
ventana = (pd.Timestamp("2026-07-06"), pd.Timestamp("2026-07-20"))
fig, ejes = plt.subplots(2, 1, figsize=(16, 8))
for eje, h in zip(ejes, HORIZONTES):
    t = prueba(h, "NORTE").reset_index(drop=True)
    objetivo = t["DateTime"] + HORIZONTES[h] * PASO
    filtro = objetivo.between(*ventana).to_numpy()
    eje.plot(objetivo[filtro], t["real"][filtro], color="black", lw=1.3, label="Real")
    eje.plot(objetivo[filtro], pred_adaptacion[h][("fijo", "M0")]["NORTE"][filtro], color=COLORES["NORTE"], lw=1,
             label="Fijo (A)")
    eje.plot(objetivo[filtro], pred_adaptacion[h][("semanal", "M0")]["NORTE"][filtro], "--", color="#7a4f00", lw=1,
             label="Semanal (C)")
    eje.plot(objetivo[filtro], ref_mejor(h, "NORTE")[filtro], ":", color="grey", lw=1,
             label=f"Mejor referencia ({NOMBRES_REF[MEJOR_REF[h]['NORTE']]})")
    eje.set_ylabel("MW")
    eje.set_title(f"Norte a {h}: dos semanas de julio de 2026")
    eje.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for eje, a in zip(ejes, AREAS):
    t = prueba("1h", a)
    residuo = t["real"].to_numpy() - pred_adaptacion["1h"][("fijo", "M0")][a]
    eje.hist(residuo, bins=80, color=COLORES[a])
    eje.axvline(0, color="black", lw=1)
    eje.set(title=f"{NOMBRES[a]} a 1 h · media {residuo.mean():.1f} · desv. {residuo.std():.1f} MW",
            xlabel="Residuo del modelo fijo: real − predicho (MW)")
plt.tight_layout()
plt.show()

# 9. Las hipótesis, una por una

Se evalúan tal como quedaron escritas en el notebook 02 (§12), antes de esta corrida.

In [ ]:
fr = frente_referencias.set_index(["Horizonte", "Área", "Método"])
fijo_A = f"{ELEGIDO}, fijo (A)"
ab = ablacion.set_index(["Horizonte", "Contraste", "Área"])
ef = efectos_adaptacion.set_index(["Horizonte", "Contraste", "Área"])


def gana(h, a):
    return fr.loc[(h, NOMBRES[a], fijo_A), "Δ MAE vs. referencia (MW)"] < 0


def texto_ic(tabla, clave):
    fila = tabla.loc[clave]
    return f"{fila['Δ MAE (MW)']:.2f} MW (IC {fila['IC 95 % bajo']:.2f} a {fila['IC 95 % alto']:.2f})"


mejora_norte = fr.loc[("1h", "Norte", fijo_A), "Mejora vs. mejor referencia (%)"]
c_a_norte = ef.loc[("1h", "Reentrenar sin exógenas (C − A)", "Norte")]
filas = [
    {"Hipótesis": "1a. A le gana a la mejor referencia a 1 h en las tres áreas",
     "¿Se cumple?": all(gana("1h", a) for a in AREAS),
     "Evidencia": ", ".join(f"{NOMBRES[a]} {fr.loc[('1h', NOMBRES[a], fijo_A), 'Mejora vs. mejor referencia (%)']:.1f} %" for a in AREAS)},
    {"Hipótesis": "1b. En el Norte la mejora a 1 h baja respecto de 2025 (49 %) pero queda sobre 25 %",
     "¿Se cumple?": 25 < mejora_norte < 49, "Evidencia": f"Norte {mejora_norte:.1f} %"},
    {"Hipótesis": "1c. A 24 h le gana en el Centro y el Norte",
     "¿Se cumple?": gana("24h", "CENTRO") and gana("24h", "NORTE"),
     "Evidencia": ", ".join(f"{NOMBRES[a]} {fr.loc[('24h', NOMBRES[a], fijo_A), 'Mejora vs. mejor referencia (%)']:.1f} %" for a in AREAS)},
    {"Hipótesis": "2. Reentrenar cada semana (C − A) en el Norte a 1 h: negativo y su IC no cruza el cero",
     "¿Se cumple?": c_a_norte["IC 95 % alto"] < 0, "Evidencia": texto_ic(ef, ("1h", "Reentrenar sin exógenas (C − A)", "Norte"))},
    {"Hipótesis": "3. M3 − M0 a 1 h no se distingue de cero en ninguna área",
     "¿Se cumple?": not any(ab.loc[("1h", "Todas (M3 − M0)", NOMBRES[a]), "Distinguible"] for a in AREAS),
     "Evidencia": "; ".join(f"{NOMBRES[a]} {texto_ic(ab, ('1h', 'Todas (M3 − M0)', NOMBRES[a]))}" for a in AREAS)},
    {"Hipótesis": "4. El clima ayuda a 24 h en el Centro (M1 − M0 negativo, IC bajo cero)",
     "¿Se cumple?": ab.loc[("24h", "Clima (M1 − M0)", "Centro"), "IC 95 % alto"] < 0,
     "Evidencia": texto_ic(ab, ("24h", "Clima (M1 − M0)", "Centro"))},
    {"Hipótesis": "5. El Niño no mejora en ningún horizonte ni área (M2 − M1 nunca con IC bajo cero)",
     "¿Se cumple?": not any(ab.loc[(h, "El Niño (M2 − M1)", NOMBRES[a]), "IC 95 % alto"] < 0 for h in HORIZONTES for a in AREAS),
     "Evidencia": "; ".join(f"{h} {NOMBRES[a]} {ab.loc[(h, 'El Niño (M2 − M1)', NOMBRES[a]), 'Δ MAE (MW)']:.2f}"
                           for h in HORIZONTES for a in AREAS)},
]
hipotesis = pd.DataFrame(filas)
hipotesis["¿Se cumple?"] = hipotesis["¿Se cumple?"].map({True: "Sí", False: "No"})
with pd.option_context("display.max_colwidth", 200):
    display(hipotesis.set_index("Hipótesis"))

# 10. Guardar

Las predicciones de 2026 de cada política y variante, por área y horizonte, quedan en `reportes/sein/prueba_2026/`
para la app y el paper, junto con las tablas de este notebook. La corrida rápida escribe en `prueba_2026_rapido/`.

In [ ]:
carpeta = RAIZ / "reportes" / "sein" / ("prueba_2026_rapido" if RAPIDO else "prueba_2026")
carpeta.mkdir(parents=True, exist_ok=True)
for h in HORIZONTES:
    for a in AREAS:
        t = prueba(h, a)
        salida = t[["DateTime", "real", *REFERENCIAS]].assign(
            **{f"{p}_{v}": pred[a] for (p, v), pred in pred_adaptacion[h].items()},
            **{f"alternativo_{ALTERNATIVO.split()[0].lower()}": alternativo[h][a]})
        salida.to_csv(carpeta / f"predicciones_{h}_{a.lower()}.csv", index=False)
for nombre, tabla in {"frente_referencias": frente_referencias, "ablacion": ablacion, "adaptacion": adaptacion,
                      "efectos_adaptacion": efectos_adaptacion, "mensual": mensual, "regimenes": regimenes_2026,
                      "hipotesis": hipotesis}.items():
    tabla.to_csv(carpeta / f"{nombre}.csv", index=False)
print("Predicciones y tablas en", carpeta)

# Conclusiones

(Se escriben con los resultados de la corrida, sin cambiar las hipótesis.)